In [2]:

from tensorflow.keras.models import load_model
model = load_model("/content/EfficientNetB1.keras")

In [3]:
# Direct download from original source (Institut für Neuroinformatik)
!wget https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip
#!wget https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_Images.zip

# Unzip both files
!unzip GTSRB_Final_Training_Images.zip
#!unzip GTSRB_Final_Test_Images.zip

# Cleanup (optional)
!rm *.zip

print("Dataset downloaded and extracted!")

Streaming output truncated to the last 5000 lines.
  inflating: GTSRB/Final_Training/Images/00035/00000_00020.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00021.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00022.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00023.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00024.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00025.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00026.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00027.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00028.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00000_00029.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00001_00000.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00001_00001.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00001_00002.ppm  
  inflating: GTSRB/Final_Training/Images/00035/00001_00003.ppm  
  inflating: GTSRB/Final_Training/Image

In [4]:
import os
from PIL import Image
import shutil

# Define paths
original_dir = '/content/GTSRB/Final_Training/Images'
processed_dir = '/content/GTSRB_Train'

# Create output directory if it doesn't exist
os.makedirs(processed_dir, exist_ok=True)

# Walk through all class folders
for class_id in os.listdir(original_dir):
    class_path = os.path.join(original_dir, class_id)
    if not os.path.isdir(class_path):
        continue  # skip non-directories

    # Create corresponding class folder in processed_dir
    output_class_path = os.path.join(processed_dir, class_id)
    os.makedirs(output_class_path, exist_ok=True)

    # Process each PPM file
    for filename in os.listdir(class_path):
        if filename.endswith('.ppm'):
            img_path = os.path.join(class_path, filename)
            img = Image.open(img_path)
            # Save as .png
            new_filename = os.path.splitext(filename)[0] + '.png'
            img.save(os.path.join(output_class_path, new_filename))

print("All .ppm files converted and saved as .png in:", processed_dir)


All .ppm files converted and saved as .png in: /content/GTSRB_Train


In [5]:
import os

# Path to the processed directory where .png files were saved
processed_dir = '/content/GTSRB_Train'

# Initialize counter
total_images = 0

# Walk through each subfolder and count .png files
for class_id in os.listdir(processed_dir):
    class_path = os.path.join(processed_dir, class_id)
    if os.path.isdir(class_path):
        num_png_files = len([f for f in os.listdir(class_path) if f.endswith('.png')])
        print(f"Class {class_id}: {num_png_files} images")
        total_images += num_png_files

print(f"\nTotal number of .png images: {total_images}")


Class 00035: 1200 images
Class 00034: 420 images
Class 00042: 240 images
Class 00016: 420 images
Class 00040: 360 images
Class 00015: 630 images
Class 00028: 540 images
Class 00041: 240 images
Class 00007: 1440 images
Class 00024: 270 images
Class 00021: 330 images
Class 00014: 780 images
Class 00009: 1470 images
Class 00037: 210 images
Class 00030: 450 images
Class 00036: 390 images
Class 00019: 210 images
Class 00011: 1320 images
Class 00022: 390 images
Class 00031: 780 images
Class 00004: 1980 images
Class 00018: 1200 images
Class 00006: 420 images
Class 00038: 2070 images
Class 00010: 2010 images
Class 00013: 2160 images
Class 00029: 270 images
Class 00001: 2220 images
Class 00039: 300 images
Class 00032: 240 images
Class 00017: 1110 images
Class 00003: 1410 images
Class 00005: 1860 images
Class 00000: 210 images
Class 00027: 240 images
Class 00025: 1500 images
Class 00012: 2100 images
Class 00002: 2250 images
Class 00023: 510 images
Class 00008: 1410 images
Class 00020: 360 images

In [6]:
import tensorflow as tf
from tensorflow.keras.models import load_model

# 1) Load your pretrained EfficientNetB1
model = load_model("/content/EfficientNetB1.keras")

# 2) Build train & validation datasets with an 80/20 split
base_dir = '/content/GTSRB_Train'
batch_size = 32
img_size   = (240, 240)
seed       = 123

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    base_dir,
    labels='inferred',
    label_mode='categorical',
    batch_size=batch_size,
    image_size=img_size,
    shuffle=True,
    validation_split=0.2,
    subset='training',
    seed=seed
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    base_dir,
    labels='inferred',
    label_mode='categorical',
    batch_size=batch_size,
    image_size=img_size,
    shuffle=False,                # no need to shuffle validation
    validation_split=0.2,
    subset='validation',
    seed=seed
)

# 3) Define your noise-fusion function (Gaussian example)
alpha_max = 0.3

def add_noise(image, label):
    alpha = tf.random.uniform([], 0.0, alpha_max)
    noise = tf.random.normal(tf.shape(image), mean=0.0, stddev=255.0)
    img_f  = tf.cast(image, tf.float32)
    fused  = img_f * (1.0 - alpha) + noise * alpha
    fused  = tf.clip_by_value(fused, 0.0, 255.0)
    return tf.cast(fused, image.dtype), label

# 4) Prepare clean and noisy training streams
clean_ds = train_ds
noisy_ds = train_ds.map(add_noise, num_parallel_calls=tf.data.AUTOTUNE)

# 5) Alternate clean ↔ noisy
paired = tf.data.Dataset.zip((clean_ds, noisy_ds))
alt_train_ds = paired.flat_map(
    lambda clean, noisy: tf.data.Dataset.from_tensors(clean)
                        .concatenate(tf.data.Dataset.from_tensors(noisy))
).prefetch(tf.data.AUTOTUNE)

# 6) Prefetch validation (no noise)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

# 7) Compile & fine-tune, including validation
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    alt_train_ds,
    epochs=10,         # or 10–20
    validation_data=val_ds
)


Found 39209 files belonging to 43 classes.
Using 31368 files for training.
Found 39209 files belonging to 43 classes.
Using 7841 files for validation.
Epoch 1/10
   1962/Unknown 502s 200ms/step - accuracy: 0.9693 - loss: 0.1132

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1962/1962 ━━━━━━━━━━━━━━━━━━━━ 526s 212ms/step - accuracy: 0.9693 - loss: 0.1131 - val_accuracy: 0.9995 - val_loss: 0.0013
Epoch 2/10
1962/1962 ━━━━━━━━━━━━━━━━━━━━ 364s 185ms/step - accuracy: 0.9895 - loss: 0.0350 - val_accuracy: 0.9999 - val_loss: 9.7739e-04
Epoch 3/10
1962/1962 ━━━━━━━━━━━━━━━━━━━━ 352s 179ms/step - accuracy: 0.9935 - loss: 0.0230 - val_accuracy: 0.9997 - val_loss: 5.9308e-04
Epoch 4/10
1962/1962 ━━━━━━━━━━━━━━━━━━━━ 352s 179ms/step - accuracy: 0.9949 - loss: 0.0170 - val_accuracy: 1.0000 - val_loss: 3.2303e-04
Epoch 5/10
1962/1962 ━━━━━━━━━━━━━━━━━━━━ 363s 185ms/step - accuracy: 0.9959 - loss: 0.0128 - val_accuracy: 1.0000 - val_loss: 1.9347e-04
Epoch 6/10
1962/1962 ━━━━━━━━━━━━━━━━━━━━ 364s 185ms/step - accuracy: 0.9964 - loss: 0.0121 - val_accuracy: 0.9999 - val_loss: 4.2409e-04
Epoch 7/10
1962/1962 ━━━━━━━━━━━━━━━━━━━━ 363s 185ms/step - accuracy: 0.9965 - loss: 0.0109 - val_accuracy: 0.9999 - val_loss: 5.1259e-04
Epoch 8/10
1962/1962 ━━━━━━━━━━━━━━━━━━━━ 363s 18

In [7]:
# 8) Save the fine-tuned model
save_path = "/content/EfficientNetB1_NFM_finetuned.keras"
model.save(save_path)
print(f"Model fine-tuned and saved at: {save_path}")

Model fine-tuned and saved at: /content/EfficientNetB1_NFM_finetuned.keras


In [8]:
import gdown
import zipfile

url = "https://drive.google.com/uc?id=1tt6pE0OpFrykOOhRRSVYfKKSQRp8MbsB"
output_path = "Fgsm_test.zip"
gdown.download(url, output_path, quiet=False)

# Unzip
with zipfile.ZipFile(output_path, 'r') as zip_ref:
    zip_ref.extractall("Fgsm_test")

Downloading...
From (original): https://drive.google.com/uc?id=1tt6pE0OpFrykOOhRRSVYfKKSQRp8MbsB
From (redirected): https://drive.google.com/uc?id=1tt6pE0OpFrykOOhRRSVYfKKSQRp8MbsB&confirm=t&uuid=b3fda52c-dc07-4e06-b58a-92594535697a
To: /content/Fgsm_test.zip
100%|██████████| 175M/175M [00:02<00:00, 84.2MB/s]


In [9]:
# Install gdown if not already installed
!pip install gdown

# Download the file from Google Drive
file_id = "1avRnoqSRSIpDiHL9_JrZe-UlE3pZwmMx"
!gdown --id {file_id}

# Check the downloaded filename (replace 'filename.zip' with the actual name)
!ls

# Unzip the file into a folder named Pgd_test
!unzip -q adversarial_images_PGD.zip -d Pgd_test  # Replace 'filename.zip' with the actual name

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1avRnoqSRSIpDiHL9_JrZe-UlE3pZwmMx
From (redirected): https://drive.google.com/uc?id=1avRnoqSRSIpDiHL9_JrZe-UlE3pZwmMx&confirm=t&uuid=5b78dc23-2e0c-48b4-bbef-70e749359c9d
To: /content/adversarial_images_PGD.zip
100% 206M/206M [00:02<00:00, 81.3MB/s]
adversarial_images_PGD.zip  EfficientNetB1_NFM_finetuned.keras	GTSRB
Attack_testdata.zip	    Fgsm_test				GTSRB_Train
EfficientNetB1.keras	    Fgsm_test.zip			sample_data


In [18]:
model_NF= load_model("/content/EfficientNetB1_NFM_finetuned.keras")

In [21]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load the CSV file
csv_path = '/content/Fgsm_test/adversarial_images/adversarial_labels.csv'
df = pd.read_csv(csv_path)

# Define the noise fusion function
def fuse_with_noise_batch(batch, alpha=0.2):
    batch = tf.cast(batch, tf.float32)
    noise = tf.random.normal(tf.shape(batch), mean=0.0, stddev=255.0)
    fused = batch * (1.0 - alpha) + noise * alpha
    fused = tf.clip_by_value(fused, 0.0, 255.0)
    return fused  # Stay float32

# List of epsilon values to evaluate
epsilons = [0.007, 0.01, 0.03, 0.1]

for eps in epsilons:
    print(f"\n{'='*50}\nEvaluating Adversarial Images for Epsilon = {eps}\n{'='*50}")

    # Filter CSV for the current epsilon
    eps_df = df[df['epsilon'] == eps].copy()

    # Remove the 'eps_X.XXX/' prefix from filenames (if present)
    eps_df['filename'] = eps_df['filename'].str.replace(f'eps_{eps}/', '', regex=False)

    # Create ImageDataGenerator (no normalization)
    test_datagen = ImageDataGenerator()

    # Prepare test generator
    test_generator = test_datagen.flow_from_dataframe(
        dataframe=eps_df,
        directory=f'/content/Fgsm_test/adversarial_images/eps_{eps}',  # Folder for current epsilon
        x_col='filename',
        y_col='label',
        target_size=(240, 240),
        batch_size=32,
        class_mode='raw',  # For integer labels
        shuffle=False
    )

    # Skip if no images found
    if test_generator.samples == 0:
        print(f" No images found for epsilon={eps}. Check paths or filenames.")
        continue

    # Predict batch-by-batch with noise fusion
    y_true = []
    y_pred = []

    for batch_x, batch_y in test_generator:
        # Apply noise fusion
        fused_batch_x = fuse_with_noise_batch(batch_x)

        # Predict
        preds = model_NF.predict(fused_batch_x, verbose=0)
        batch_preds = tf.argmax(preds, axis=1).numpy()

        y_true.append(batch_y)
        y_pred.append(batch_preds)

        # Exit when all samples processed
        if len(np.concatenate(y_true)) >= test_generator.samples:
            break

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    # Calculate metrics
    acc = accuracy_score(y_true, y_pred)
    print(f"\nPost-Adversarial Accuracy (with Noise Fusion, ε={eps}): {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))



Evaluating Adversarial Images for Epsilon = 0.007
Found 516 validated image filenames.

Post-Adversarial Accuracy (with Noise Fusion, ε=0.007): 0.9477

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00        12
           2       1.00      0.92      0.96        12
           3       0.79      0.92      0.85        12
           4       1.00      1.00      1.00        12
           5       0.85      0.92      0.88        12
           6       1.00      0.67      0.80        12
           7       1.00      1.00      1.00        12
           8       0.92      0.92      0.92        12
           9       1.00      0.92      0.96        12
          10       1.00      0.92      0.96        12
          11       0.80      1.00      0.89        12
          12       1.00      0.92      0.96        12
          13       1.00      1.00      1.00        12
          14 

In [22]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load the CSV file
csv_path = '/content/Pgd_test/adversarial_images_PGD/adversarial_labels.csv'
df = pd.read_csv(csv_path)

# Define the noise fusion function
def fuse_with_noise_batch(batch, alpha=0.2):
    batch = tf.cast(batch, tf.float32)
    noise = tf.random.normal(tf.shape(batch), mean=0.0, stddev=255.0)
    fused = batch * (1.0 - alpha) + noise * alpha
    fused = tf.clip_by_value(fused, 0.0, 255.0)
    return fused  # Stay float32

# List of epsilon values to evaluate
epsilons = [0.007, 0.01, 0.03, 0.1]

for eps in epsilons:
    print(f"\n{'='*50}\nEvaluating Adversarial Images for Epsilon = {eps}\n{'='*50}")

    # Filter CSV for the current epsilon
    eps_df = df[df['epsilon'] == eps].copy()

    # Remove the 'eps_X.XXX/' prefix from filenames (if present)
    eps_df['filename'] = eps_df['filename'].str.replace(f'eps_{eps}/', '', regex=False)

    # Create ImageDataGenerator (no normalization)
    test_datagen = ImageDataGenerator()

    # Prepare test generator
    test_generator = test_datagen.flow_from_dataframe(
        dataframe=eps_df,
        directory=f'/content/Pgd_test/adversarial_images_PGD/eps_{eps}',  # Folder for current epsilon
        x_col='filename',
        y_col='label',
        target_size=(240, 240),
        batch_size=32,
        class_mode='raw',  # For integer labels
        shuffle=False
    )

    # Skip if no images found
    if test_generator.samples == 0:
        print(f" No images found for epsilon={eps}. Check paths or filenames.")
        continue

    # Predict batch-by-batch with noise fusion
    y_true = []
    y_pred = []

    for batch_x, batch_y in test_generator:
        # Apply noise fusion
        fused_batch_x = fuse_with_noise_batch(batch_x)

        # Predict
        preds = model_NF.predict(fused_batch_x, verbose=0)
        batch_preds = tf.argmax(preds, axis=1).numpy()

        y_true.append(batch_y)
        y_pred.append(batch_preds)

        # Exit when all samples processed
        if len(np.concatenate(y_true)) >= test_generator.samples:
            break

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    # Calculate metrics
    acc = accuracy_score(y_true, y_pred)
    print(f"\nPost-Adversarial Accuracy (with Noise Fusion, ε={eps}): {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))



Evaluating Adversarial Images for Epsilon = 0.007
Found 516 validated image filenames.

Post-Adversarial Accuracy (with Noise Fusion, ε=0.007): 0.9419

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00        12
           2       1.00      0.92      0.96        12
           3       0.69      0.92      0.79        12
           4       1.00      1.00      1.00        12
           5       0.79      0.92      0.85        12
           6       1.00      0.58      0.74        12
           7       1.00      1.00      1.00        12
           8       0.92      0.92      0.92        12
           9       1.00      0.92      0.96        12
          10       1.00      0.92      0.96        12
          11       0.75      1.00      0.86        12
          12       0.92      0.92      0.92        12
          13       1.00      1.00      1.00        12
          14 

| **Attack**       | **Epsilon (ε)** | **Alpha (α)** | **Test Accuracy after applying attack** | **Accuracy after Noise Fusion Defensive Mechanism** |
|:----------------:|:---------------:|:-------------:|:----------------------------:|:-------------------------------:|
| **Original Model**| -               | -             | 96%                          | -                               |
| **FGSM**         | 0.007           | -             | 58%                          | 94.77%                          |
| **FGSM**         | 0.01            | -             | 44%                          | 93.80%                          |
| **FGSM**         | 0.03            | -             | 14%                          | 84.11%                          |
| **FGSM**         | 0.1             | -             | 6%                           | 52.52%                          |
| **PGD**          | 0.007           | 0.00175       | 35%                          | 94.19%                          |
| **PGD**          | 0.01            | 0.0025        | 7%                           | 86.05%                          |
| **PGD**          | 0.03            | 0.0075        | 3%                           | 78.68%                          |
| **PGD**          | 0.1             | 0.025         | 0%                           | 59.30%                          |
